# 📚 Python 双指针 & 滑动窗口 — LeetCode 复习笔记

> **覆盖题目**：LC 167 · LC 11 · LC 26 · LC 141（对照）· LC 15 · LC 18 · LC 209

| 编号 | 题名 | 题型 | 难度 |
|------|------|------|------|
| 167 | 两数之和 II | 对撞双指针 | 🟡 |
| 11  | 盛最多水的容器 | 贪心双指针 | 🟡 |
| 26  | 删除有序数组中的重复项 | 读写双指针（快慢）| 🟢 |
| 141 | 环形链表（对照）| 追及双指针（快慢）| 🟢 |
| 15  | 三数之和 | 排序 + 对撞双指针 | 🟡 |
| 18  | 四数之和 | 排序 + 双层循环 + 对撞双指针 | 🟡 |
| 209 | 长度最小的子数组 | 变长滑动窗口 | 🟡 |

In [ ]:
from typing import List
import pandas as pd
print("环境就绪 ✅")

---
## 1. LC 167 — 两数之和 II（对撞双指针）

| 项目 | 内容 |
|------|------|
| **题型** | 对撞双指针（Converging Two Pointers）|
| **时间复杂度** | O(n) |
| **空间复杂度** | O(1) |
| **数据范围** | 2 ≤ n ≤ 3×10⁴，−1000 ≤ nums[i] ≤ 1000，数组非递减，唯一解 |

### ⚡ 触发条件
1. 数组**已排序**（非递减/非递增）
2. 寻找**两个元素**之和满足条件
3. 要求 **O(1) 额外空间**（排除哈希表）

### 🧠 算法
```
left, right = 0, n-1
while left < right:
    s = nums[left] + nums[right]
    if s == target → 返回 [left+1, right+1]（1-indexed）
    elif s < target → left += 1   # 需要更大
    else           → right -= 1  # 需要更小
```
**正确性**：排序单调性保证每步操作永久淘汰一个元素，共 O(n) 步。

In [ ]:
# 模板：对撞双指针（有序数组 + 找两元素之和）
def converging_two_ptr(nums: List[int], target: int) -> list:
    left, right = 0, len(nums) - 1
    while left < right:
        s = nums[left] + nums[right]
        if s == target:   return [left, right]   # 按题目调整 0/1-indexed
        elif s < target:  left += 1
        else:             right -= 1
    return []
print("✅ 对撞双指针模板")

In [ ]:
class Solution_167:
    def twoSum(self, numbers: List[int], target: int) -> List[int]:
        left, right = 0, len(numbers) - 1
        while left < right:
            s = numbers[left] + numbers[right]
            if s == target:   return [left + 1, right + 1]
            elif s < target:  left += 1
            else:             right -= 1
        return []

sol = Solution_167()
cases = [([2,7,11,15],9,[1,2]),([2,3,4],6,[1,3]),([-1,0],-1,[1,2])]
for nums,t,e in cases:
    r = sol.twoSum(nums,t)
    print(f"{'✅' if r==e else '❌'}  {nums}, target={t} → {r}")

---
## 2. LC 11 — 盛最多水的容器（贪心双指针）

| 项目 | 内容 |
|------|------|
| **题型** | 贪心双指针（Greedy Two Pointers）|
| **时间复杂度** | O(n) |
| **空间复杂度** | O(1) |
| **数据范围** | 2 ≤ n ≤ 10⁵，0 ≤ height[i] ≤ 10⁴ |

### ⚡ 触发条件
1. 选两个索引 `l < r`，最大化 `(r-l) × min(h[l], h[r])`
2. 暴力 O(n²) 超时，需线性优化
3. 存在贪心剪枝：**矮侧是瓶颈**，移走矮侧才有改善可能

> 与 LC 167 区别：LC 167 靠**有序性**，LC 11 靠**贪心剪枝**

### 🧠 算法
```
left, right = 0, n-1, max_area = 0
while left < right:
    area = (right-left) × min(h[left], h[right])
    max_area = max(max_area, area)
    if h[left] <= h[right]: left += 1   # 移走矮侧
    else:                   right -= 1
```
**贪心证明**：设 h[l]≤h[r]，移右侧(较高)时，宽-1且高上限不变，面积不可能增大；
移左侧(较矮)时，有机会找到更高的柱子。∴ 移矮侧是唯一可能改善的操作。

In [ ]:
class Solution_11:
    def maxArea(self, height: List[int]) -> int:
        left, right, best = 0, len(height)-1, 0
        while left < right:
            best = max(best, (right-left)*min(height[left], height[right]))
            if height[left] <= height[right]: left += 1 #注意是左右指针对应的值比，而不是左指针和左指针后一个值，右指针和右指针的一个值比
            else:                             right -= 1
        return best

sol = Solution_11()
cases = [([1,8,6,2,5,4,8,3,7],49),([1,1],1),([4,3,2,1,4],16)]
for h,e in cases:
    r = sol.maxArea(h)
    print(f"{'✅' if r==e else '❌'}  height={h} → {r} (期望 {e})")

---
## 3. LC 26 + LC 141 — 快慢双指针（读写 vs 追及）

### 快慢指针两种范式

| 范式 | 代表题 | slow 角色 | fast 角色 | 速度关系 | 目的 |
|------|--------|-----------|-----------|---------|------|
| **读写双指针** | LC 26, 80, 283 | 写指针（有效区末尾）| 读指针（扫描位置）| fast 恒前进，slow **条件性**前进 | 原地筛选数组 |
| **追及双指针** | LC 141, 142, 876 | 慢指针（1步/次）| 快指针（2步/次）| fast **固定**比 slow 快 1步 | 判环/找中点 |

---
### LC 26 — 删除有序数组中的重复项

| 项目 | 内容 |
|------|------|
| **题型** | 读写双指针（快慢） |
| **时间复杂度** | O(n) |
| **空间复杂度** | O(1) |
| **数据范围** | 1 ≤ n ≤ 3×10⁴，−100 ≤ nums[i] ≤ 100，非递减有序 |

### ⚡ 触发条件
1. 原地修改数组，O(1) 空间
2. 需要跳过重复/特定值（slow 维护有效区，fast 扫描）
3. 有序数组，重复元素必相邻

### 🧠 算法
```
slow = 0          # 写指针：有效区末尾
for fast in range(1, n):
    if nums[fast] != nums[slow]:    # 发现新的唯一元素
        slow += 1
        nums[slow] = nums[fast]
return slow + 1
```

---
### LC 141 — 环形链表（Floyd 判圈，对照）

| 项目 | 内容 |
|------|------|
| **题型** | 追及双指针（快慢） |
| **时间复杂度** | O(n) |
| **空间复杂度** | O(1) |
| **数据范围** | 0 ≤ 节点数 ≤ 10⁴ |

### ⚡ 触发条件
1. 链表判环 / 找链表中点 / 找环入口
2. 要求 O(1) 空间（排除哈希集合法）

### 🧠 算法
```
slow, fast = head, head
while fast and fast.next:
    slow = slow.next        # 1步
    fast = fast.next.next   # 2步
    if slow is fast: return True   # 有环（注意用 is，不用 ==）
return False
```

In [ ]:
# ── 读写双指针模板 ──────────────────────────────────────────
# keep_condition 变体：
#   LC 26  (去重保留1次): nums[fast] != nums[slow]
#   LC 80  (去重保留2次): nums[fast] != nums[slow-1]
#   LC 283 (移零到末尾):  nums[fast] != 0

class Solution_26:
    def removeDuplicates(self, nums: List[int]) -> int:
        slow = 0
        for fast in range(1, len(nums)):
            if nums[fast] != nums[slow]:
                slow += 1
                nums[slow] = nums[fast]
        return slow + 1

# ── 追及双指针（Floyd 判圈）───────────────────────────────────
class ListNode:
    def __init__(self, v=0): self.val=v; self.next=None

class Solution_141:
    def hasCycle(self, head: ListNode) -> bool:
        slow, fast = head, head
        while fast and fast.next:
            slow = slow.next
            fast = fast.next.next
            if slow is fast: return True
        return False

def make_list(vals, pos=-1):
    if not vals: return None
    nodes = [ListNode(v) for v in vals]
    for i in range(len(nodes)-1): nodes[i].next = nodes[i+1]
    if pos >= 0: nodes[-1].next = nodes[pos]
    return nodes[0]

print("✅ LC 26 + LC 141 解法已加载")

In [ ]:
# ── 测试 LC 26 ──
s26 = Solution_26()
for nums, ek, ev in [([1,1,2],2,[1,2]),([0,0,1,1,1,2,2,3],4,[0,1,2,3]),([1],1,[1])]:
    nc = nums.copy(); k = s26.removeDuplicates(nc)
    ok = k==ek and nc[:k]==ev
    print(f"{'✅' if ok else '❌'}  {nums} → k={k}, 前{k}项={nc[:k]}")

print()
# ── 测试 LC 141 ──
s141 = Solution_141()
for vals, pos, exp in [([3,1,2,0],1,True),([1,2],-1,False),([1],0,True),([],-1,False)]:
    r = s141.hasCycle(make_list(vals, pos))
    desc = f"尾→{pos}" if pos>=0 else "无环"
    print(f"{'✅' if r==exp else '❌'}  {vals} ({desc}) → {r}")

---
## 4. LC 15 — 三数之和（排序 + 对撞双指针）

| 项目 | 内容 |
|------|------|
| **题型** | 排序 + 对撞双指针（kSum 降维：3→2）|
| **时间复杂度** | O(n²)：排序 O(n log n) + 双层循环 O(n²) |
| **空间复杂度** | O(1)（不计输出）|
| **数据范围** | 3 ≤ n ≤ 3000，−10⁵ ≤ nums[i] ≤ 10⁵ |

### ⚡ 触发条件
1. 找**三个元素**之和 = target，结果**不含重复三元组**
2. 暴力 O(n³) 超时，需降维
3. 允许排序

### 🧠 算法（降维：固定 i，内层转 LC 167）
```
nums.sort()
for i in range(n):
    if nums[i] > 0: break                         # 剪枝
    if i > 0 and nums[i] == nums[i-1]: continue   # 去重①（外层 i，向前比较！）
    left, right = i+1, n-1
    while left < right:
        total = nums[i]+nums[left]+nums[right]
        if total == 0:
            记录三元组
            while nums[left]==nums[left+1]: left += 1   # 去重②
            while nums[right]==nums[right-1]: right -= 1 # 去重③
            left += 1; right -= 1
        elif total < 0: left += 1
        else:           right -= 1
```
> ⚠️ 外层去重向**前**比 `i-1`，不向后 `i+1`（i+1 尚未处理，不能跳过）

In [ ]:
class Solution_15:
    def threeSum(self, nums: List[int]) -> List[List[int]]:
        nums = sorted(nums); res = []; n = len(nums)
        for i in range(n):
            if nums[i] > 0: break
            if i > 0 and nums[i] == nums[i-1]: continue
            left, right = i+1, n-1
            while left < right:
                total = nums[i]+nums[left]+nums[right]
                if total == 0:
                    res.append([nums[i], nums[left], nums[right]])
                    while left < right and nums[left] == nums[left+1]:   left += 1
                    while left < right and nums[right] == nums[right-1]: right -= 1
                    left += 1; right -= 1
                elif total < 0: left += 1
                else:           right -= 1
        return res

def norm(t): return sorted([sorted(x) for x in t])
sol15 = Solution_15()
cases = [([-1,0,1,2,-1,-4],[[-1,-1,2],[-1,0,1]]),([0,1,1],[]),([0,0,0],[[0,0,0]]),([-2,0,0,2,2],[[-2,0,2]])]
for nums, exp in cases:
    r = sol15.threeSum(nums.copy())
    ok = norm(r)==norm(exp)
    print(f"{'✅' if ok else '❌'}  {nums} → {norm(r)}")

---
## 5. LC 18 — 四数之和（排序 + 双层循环 + 对撞双指针）

| 项目 | 内容 |
|------|------|
| **题型** | 排序 + 双层循环 + 对撞双指针（kSum 降维：4→2）|
| **时间复杂度** | O(n³) |
| **空间复杂度** | O(1)（不计输出）|
| **数据范围** | 1 ≤ n ≤ 200，−10⁹ ≤ nums[i], target ≤ 10⁹ |

### ⚡ 触发条件
1. 找**四个元素**之和 = target，结果**不含重复四元组**
2. n 较小（≤200），O(n³) 可接受
3. 可以排序

### 🧠 算法（kSum 通用降维）

| k | 外层固定 | 时间 |
|---|---------|------|
| 2 (LC 167) | 0层 | O(n) |
| 3 (LC 15)  | 1层 | O(n²) |
| 4 (LC 18)  | 2层 | O(n³) |

**双重剪枝**（排序后通用）：
- `最小和 > target` → **break**（后续更大，直接终止）
- `最大和 < target` → **continue**（当前太小，换更大的 i/j）

In [ ]:
class Solution_18:
    def fourSum(self, nums: List[int], target: int) -> List[List[int]]:
        if len(nums) < 4: return []
        nums.sort(); res = []; n = len(nums)
        for i in range(n-3):
            if i > 0 and nums[i] == nums[i-1]: continue
            if nums[i]+nums[i+1]+nums[i+2]+nums[i+3] > target: break    # 剪枝
            if nums[i]+nums[n-3]+nums[n-2]+nums[n-1] < target: continue # 剪枝
            for j in range(i+1, n-2):
                if j > i+1 and nums[j] == nums[j-1]: continue
                if nums[i]+nums[j]+nums[j+1]+nums[j+2] > target: break
                if nums[i]+nums[j]+nums[n-2]+nums[n-1] < target: continue
                left, right = j+1, n-1
                while left < right:
                    total = nums[i]+nums[j]+nums[left]+nums[right]
                    if total == target:
                        res.append([nums[i],nums[j],nums[left],nums[right]])
                        while left < right and nums[left]==nums[left+1]:   left += 1
                        while left < right and nums[right]==nums[right-1]: right -= 1
                        left += 1; right -= 1
                    elif total < target: left += 1
                    else:                right -= 1
        return res

def norm4(q): return sorted([sorted(x) for x in q])
sol18 = Solution_18()
cases = [([1,0,-1,0,-2,2],0,[[-2,-1,1,2],[-2,0,0,2],[-1,0,0,1]]),
         ([2,2,2,2,2],8,[[2,2,2,2]]),([0,0,0,0],0,[[0,0,0,0]])]
for nums, t, exp in cases:
    r = sol18.fourSum(nums.copy(), t)
    ok = norm4(r)==norm4(exp)
    print(f"{'✅' if ok else '❌'}  target={t}, nums={nums} → {len(r)}组")

---
## 6. LC 209 — 长度最小的子数组（变长滑动窗口）

| 项目 | 内容 |
|------|------|
| **题型** | 变长滑动窗口（Variable-size Sliding Window）|
| **时间复杂度** | O(n)（每个元素均摊进出各一次）|
| **空间复杂度** | O(1) |
| **数据范围** | 1 ≤ n ≤ 10⁵，1 ≤ target ≤ 10⁹，1 ≤ nums[i] ≤ 10⁴（**全为正数**）|

### ⚡ 触发条件
1. 求**连续子数组**满足某条件
2. 数组元素**全为正数（或非负）** → 保证窗口和的单调性，才能安全收缩
3. 求满足条件的**最短 / 最长**长度
4. 暴力 O(n²) 超时

> ⚠️ 若数组含负数，滑动窗口**失效**（收缩左边界时总和可能反而变大），需改用前缀和。

### 🧠 算法（"求最短"型）
```
lk, total, min_len = 0, 0, ∞
for rk in range(n):          # 右指针扩张（"吸气"）
    total += nums[rk]
    while total >= target:   # 条件满足 → 收缩左边界（"呼气"，用 while 不是 if！）
        min_len = min(min_len, rk - lk + 1)
        total -= nums[lk]
        lk += 1
return min_len if min_len != ∞ else 0
```
> `while` 而非 `if`：一次右移可能让窗口"超额"满足，需持续收缩到刚好不满足为止。

In [ ]:
class Solution_209:
    def minSubArrayLen(self, target: int, nums: List[int]) -> int:
        n = len(nums)
        lk, total, min_len = 0, 0, float('inf')
        for rk in range(n):
            total += nums[rk]
            while total >= target:
                min_len = min(min_len, rk - lk + 1)
                total -= nums[lk]
                lk += 1
        return min_len if min_len != float('inf') else 0

sol209 = Solution_209()
cases = [(7,[2,3,1,2,4,3],2),(4,[1,4,4],1),(11,[1,1,1,1,1,1,1,1],0),(15,[1,2,3,4,5],5)]
for t, nums, exp in cases:
    r = sol209.minSubArrayLen(t, nums)
    print(f"{'✅' if r==exp else '❌'}  target={t}, {nums} → {r} (期望 {exp})")

---
## 📋 总结：双指针 & 滑动窗口全模板

### 题型对照表

| 题号 | 题型 | 时间 | 空间 | 关键触发 | 核心决策 | 易错点 |
|------|------|------|------|---------|---------|-------|
| 167 | 对撞双指针 | O(n) | O(1) | 已排序+两元素和+O(1)空间 | s<t:left++; s>t:right-- | 1-indexed返回 |
| 11  | 贪心双指针 | O(n) | O(1) | 最大化宽×矮高 | 移矮侧 | 相等时移哪侧均可 |
| 26  | 读写双指针 | O(n) | O(1) | 原地去重/筛选+有序 | nums[fast]!=nums[slow] → slow++ | return slow+1 |
| 141 | 追及双指针 | O(n) | O(1) | 链表判环/找中点 | slow is fast → 有环 | 用 is 非 == |
| 15  | 排序+对撞 | O(n²) | O(1) | 三元素和+不重复 | 固定i，内层LC167 | 外层去重向前比(i-1) |
| 18  | 排序+双层+对撞 | O(n³) | O(1) | 四元素和+不重复 | break(下界)/continue(上界) | break vs continue 方向 |
| 209 | 变长滑动窗口 | O(n) | O(1) | 连续子数组+全正数+最短 | while收缩，不是if | 含负数时失效 |

---

### 🔁 所有模板

```python
# 1. 对撞双指针
left, right = 0, n-1
while left < right:
    s = f(arr[left], arr[right])
    if s==target: pass          # 记录
    elif s<target: left += 1
    else:          right -= 1

# 2. 贪心双指针
left, right = 0, n-1; best = 0
while left < right:
    best = max(best, f(arr[left], arr[right]))
    if arr[left] <= arr[right]: left += 1
    else:                       right -= 1

# 3. 读写双指针（keep 可换条件）
slow = 0
for fast in range(1, n):
    if keep(arr[fast], arr[slow]):
        slow += 1; arr[slow] = arr[fast]
return slow + 1

# 4. 追及双指针（链表判环）
slow, fast = head, head
while fast and fast.next:
    slow, fast = slow.next, fast.next.next
    if slow is fast: return True
return False

# 5. 排序+对撞（kSum核心，外层加循环即可扩展到4Sum）
nums.sort()
for i in range(n):
    if nums[i]>0: break
    if i>0 and nums[i]==nums[i-1]: continue
    left, right = i+1, n-1
    while left < right:
        total = nums[i]+nums[left]+nums[right]
        if total==0:
            res.append([nums[i],nums[left],nums[right]])
            while left<right and nums[left]==nums[left+1]:   left+=1
            while left<right and nums[right]==nums[right-1]: right-=1
            left+=1; right-=1
        elif total<0: left+=1
        else:         right-=1

# 6. 变长滑动窗口（求最短）
lk, total, ans = 0, 0, float('inf')
for rk in range(n):
    total += arr[rk]
    while total >= target:
        ans = min(ans, rk-lk+1); total -= arr[lk]; lk += 1
return ans if ans!=float('inf') else 0
```